# Ensemble V4: Logit-Level Fusion

This notebook tests whether ensembling before the softmax layer improves performance.

Compared methods:

- probability soft vote baselines;
- mean logits;
- fixed weighted logits;
- validation-optimized weighted logits;
- z-normalized logits;
- temperature-scaled logits.

All model combinations and optimized weights are selected using validation Macro F1 only. The official test set is only used for the validation-selected method finalists.

## 1. Setup And Candidate Models

In [ ]:
import itertools
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data")
TRAIN_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "b. Testing Set"

IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"
NUM_CLASSES = 5
IMAGE_SIZE = 384
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42
VAL_SIZE = 0.2
DEVICE_MODE = "auto"

MIN_ENSEMBLE_SIZE = 3
MAX_ENSEMBLE_SIZE = 7
MODEL_WEIGHT_RANDOM_TRIALS = 300

VALIDATION_SEARCH_CSV = "/kaggle/working/ensemble_logit_v1_validation_search.csv"
METHOD_FINALISTS_CSV = "/kaggle/working/ensemble_logit_v1_method_finalists.csv"
INDIVIDUAL_VALIDATION_CSV = "/kaggle/working/ensemble_logit_v1_individual_validation_metrics.csv"
FINALIST_PREDICTIONS_CSV = "/kaggle/working/ensemble_logit_v1_finalist_predictions.csv"
TEMPERATURES_CSV = "/kaggle/working/ensemble_logit_v1_temperatures.csv"

# Same candidates as the final V3 ensemble. Base weights come from validation Macro F1
# and are used only for the fixed weighted variants.
MODEL_SPECS = [
    {"run_id":"B2-4", "model":"efficientnet_b2", "checkpoint_filename":"b2_4_best_efficientnet_b2.pth", "preprocessing":"none", "hidden_dim":256, "dropout":0.20, "base_weight":0.576730},
    {"run_id":"B2-5", "model":"efficientnet_b2", "checkpoint_filename":"b2_5_best_efficientnet_b2.pth", "preprocessing":"none", "hidden_dim":256, "dropout":0.20, "base_weight":0.657509},
    {"run_id":"B2-6", "model":"efficientnet_b2", "checkpoint_filename":"b2_6_best_efficientnet_b2.pth", "preprocessing":"none", "hidden_dim":256, "dropout":0.20, "base_weight":0.629050},
    {"run_id":"R7", "model":"resnet18", "checkpoint_filename":"r7_best_resnet18.pth", "preprocessing":"crop_graham", "hidden_dim":512, "dropout":0.20, "base_weight":0.614526},
    {"run_id":"B0-7", "model":"efficientnet_b0", "checkpoint_filename":"b0_7_best_efficientnet_b0.pth", "preprocessing":"crop_graham", "hidden_dim":256, "dropout":0.20, "base_weight":0.575206},
    {"run_id":"B0-6", "model":"efficientnet_b0", "checkpoint_filename":"b0_6_best_efficientnet_b0.pth", "preprocessing":"crop_graham", "hidden_dim":256, "dropout":0.20, "base_weight":0.585416},
    {"run_id":"B2-4-GGC", "model":"efficientnet_b2", "checkpoint_filename":"confirm_b2-4_gray_gamma_clahe_best_efficientnet_b2.pth", "preprocessing":"gray_gamma_clahe", "hidden_dim":256, "dropout":0.20, "base_weight":0.591841},
    {"run_id":"B2-6-GGC", "model":"efficientnet_b2", "checkpoint_filename":"confirm_b2-6_gray_gamma_clahe_best_efficientnet_b2.pth", "preprocessing":"gray_gamma_clahe", "hidden_dim":256, "dropout":0.20, "base_weight":0.599034},
    {"run_id":"B2-4-GCBM", "model":"efficientnet_b2", "checkpoint_filename":"confirm_b2-4_green_clahe_bgsub_mask_best_efficientnet_b2.pth", "preprocessing":"green_clahe_bgsub_mask", "hidden_dim":256, "dropout":0.20, "base_weight":0.615152},
]


def get_device():
    if DEVICE_MODE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable. Set DEVICE_MODE='auto' or attach a GPU.")
        return torch.device("cuda")
    if DEVICE_MODE == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device("cpu")


DEVICE = get_device()
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Candidates:", [spec["run_id"] for spec in MODEL_SPECS])

## 2. Data And Preprocessing

In [ ]:
train_full_df = pd.read_csv(TRAIN_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
test_df = pd.read_csv(TEST_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
train_full_df[LABEL_COL] = train_full_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

_, val_df = train_test_split(
    train_full_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_full_df[LABEL_COL],
)

print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

In [ ]:
def resolve_image_path(image_dir, image_name):
    image_dir = Path(image_dir)
    image_name = str(image_name)
    direct = image_dir / image_name
    if direct.exists():
        return direct
    for extension in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]:
        candidate = image_dir / f"{image_name}{extension}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Image not found: {image_name}")


def crop_black_background(image, threshold=10, padding=16):
    gray = np.array(image.convert("L"))
    coordinates = np.argwhere(gray > threshold)
    if coordinates.size == 0:
        return image
    y0, x0 = coordinates.min(axis=0)
    y1, x1 = coordinates.max(axis=0) + 1
    return image.crop((
        max(int(x0) - padding, 0),
        max(int(y0) - padding, 0),
        min(int(x1) + padding, image.width),
        min(int(y1) + padding, image.height),
    ))


def crop_graham(image):
    image = crop_black_background(image)
    array = np.array(image)
    blurred = cv2.GaussianBlur(array, (0, 0), 10)
    enhanced = cv2.addWeighted(array, 4.0, blurred, -4.0, 128)
    return Image.fromarray(np.clip(enhanced, 0, 255).astype(np.uint8))


def gray_gamma_clahe(image, gamma=1.2):
    array = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)
    inverse_gamma = 1.0 / gamma
    table = np.array([((value / 255.0) ** inverse_gamma) * 255 for value in range(256)]).astype("uint8")
    corrected = cv2.LUT(gray, table)
    enhanced = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8)).apply(corrected)
    return Image.fromarray(np.stack([enhanced, enhanced, enhanced], axis=-1))


def green_clahe_bgsub_mask(image, sigma=15):
    array = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    green = array[:, :, 1]
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(green)
    background = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=sigma)
    equalized = cv2.addWeighted(enhanced, 4, background, -4, 128)
    cleaned = cv2.bitwise_and(equalized, equalized, mask=mask)
    return Image.fromarray(np.stack([cleaned, cleaned, cleaned], axis=-1))


def preprocess_image(image, preset):
    if preset == "none":
        return image
    if preset == "crop_graham":
        return crop_graham(image)
    if preset == "gray_gamma_clahe":
        return gray_gamma_clahe(image)
    if preset == "green_clahe_bgsub_mask":
        return green_clahe_bgsub_mask(image)
    raise ValueError(f"Unsupported preprocessing: {preset}")


class FundusEvalDataset(Dataset):
    def __init__(self, dataframe, image_dir, preprocessing):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.preprocessing = preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        path = resolve_image_path(self.image_dir, row[IMAGE_COL])
        image = preprocess_image(Image.open(path).convert("RGB"), self.preprocessing)
        return self.transform(image), int(row[LABEL_COL]), path.name


def make_loader(dataframe, image_dir, preprocessing):
    dataset = FundusEvalDataset(dataframe, image_dir, preprocessing)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )

## 3. Load Models And Cache Logits

In [ ]:
def classification_head(in_features, hidden_dim, dropout, activation):
    activation_layer = nn.ReLU(inplace=True) if activation == "relu" else nn.SiLU(inplace=True)
    return nn.Sequential(
        nn.Linear(in_features, hidden_dim),
        activation_layer,
        nn.BatchNorm1d(hidden_dim),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, NUM_CLASSES),
    )


def create_model(spec):
    if spec["model"] == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = classification_head(model.fc.in_features, spec["hidden_dim"], spec["dropout"], "relu")
        return model
    if spec["model"] == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier = classification_head(model.classifier[-1].in_features, spec["hidden_dim"], spec["dropout"], "silu")
        return model
    if spec["model"] == "efficientnet_b2":
        model = models.efficientnet_b2(weights=None)
        model.classifier = classification_head(model.classifier[-1].in_features, spec["hidden_dim"], spec["dropout"], "silu")
        return model
    raise ValueError(spec["model"])


def find_checkpoint(filename):
    matches = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Checkpoint not found: {filename}. Attach the Kaggle notebook output containing this file."
        )
    return sorted(matches, key=str)[0]


def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        return checkpoint["state_dict"]
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def load_model(spec):
    path = find_checkpoint(spec["checkpoint_filename"])
    model = create_model(spec)
    model.load_state_dict(load_state(path))
    model.to(DEVICE).eval()
    print(f"Loaded {spec['run_id']}: {path}")
    return model, path


def metric_dict(labels, predictions):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(labels, predictions, weights="quadratic"),
    }


@torch.no_grad()
def predict_logits(model, loader):
    logits_all, probabilities_all, predictions_all = [], [], []
    labels_all, names_all = [], []
    for images, labels, names in tqdm(loader, desc="Predicting", leave=False):
        logits = model(images.to(DEVICE))
        probabilities = torch.softmax(logits, dim=1)
        logits_all.append(logits.cpu().numpy())
        probabilities_all.append(probabilities.cpu().numpy())
        predictions_all.append(logits.argmax(dim=1).cpu().numpy())
        labels_all.extend(labels.numpy())
        names_all.extend(list(names))
    return (
        np.concatenate(logits_all),
        np.concatenate(probabilities_all),
        np.concatenate(predictions_all),
        np.asarray(labels_all),
        names_all,
    )

In [ ]:
def collect_cache(dataframe, image_dir, record_validation_metrics=False):
    logits_all, probabilities_all, predictions_all = [], [], []
    labels_reference, names_reference = None, None
    individual_rows = []

    for spec in MODEL_SPECS:
        model, checkpoint_path = load_model(spec)
        loader = make_loader(dataframe, image_dir, spec["preprocessing"])
        logits, probabilities, predictions, labels, names = predict_logits(model, loader)

        if labels_reference is None:
            labels_reference, names_reference = labels, names
        else:
            assert np.array_equal(labels_reference, labels), "Label order mismatch"
            assert names_reference == names, "Image-name order mismatch"

        if record_validation_metrics:
            result = metric_dict(labels, predictions)
            individual_rows.append({
                "run_id": spec["run_id"],
                "model": spec["model"],
                "preprocessing": spec["preprocessing"],
                "checkpoint_path": str(checkpoint_path),
                "val_accuracy": result["accuracy"],
                "val_macro_f1": result["macro_f1"],
                "val_kappa": result["kappa"],
                "logit_mean": float(logits.mean()),
                "logit_std": float(logits.std()),
            })

        logits_all.append(logits)
        probabilities_all.append(probabilities)
        predictions_all.append(predictions)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return {
        "logits": np.stack(logits_all),                 # [models, samples, classes]
        "probabilities": np.stack(probabilities_all),
        "predictions": np.stack(predictions_all),
        "labels": labels_reference,
        "names": names_reference,
        "individual": pd.DataFrame(individual_rows),
    }


val_cache = collect_cache(val_df, TRAIN_IMAGE_DIR, record_validation_metrics=True)
test_cache = collect_cache(test_df, TEST_IMAGE_DIR, record_validation_metrics=False)
val_cache["individual"].to_csv(INDIVIDUAL_VALIDATION_CSV, index=False)
display(val_cache["individual"].sort_values("val_macro_f1", ascending=False))

## 4. Validation-Fitted Logit Calibration

In [ ]:
def fit_temperature_for_model(logits, labels, max_iter=200):
    # Fits one positive temperature per model on validation NLL.
    logits_tensor = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    labels_tensor = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    log_temperature = torch.zeros(1, device=DEVICE, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.05, max_iter=max_iter, line_search_fn="strong_wolfe")
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature).clamp(0.05, 20.0)
        loss = criterion(logits_tensor / temperature, labels_tensor)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(log_temperature).detach().clamp(0.05, 20.0).cpu().item())


def fit_logit_transforms(val_logits, labels):
    # Z-normalization parameters are fitted on validation logits and reused for test.
    z_mean = val_logits.mean(axis=(1, 2), keepdims=True)
    z_std = val_logits.std(axis=(1, 2), keepdims=True)
    z_std = np.clip(z_std, 1e-6, None)

    temperatures = []
    for model_index in range(val_logits.shape[0]):
        temperatures.append(fit_temperature_for_model(val_logits[model_index], labels))
    temperatures = np.asarray(temperatures, dtype=np.float32).reshape(-1, 1, 1)
    return z_mean, z_std, temperatures


Z_MEAN, Z_STD, TEMPERATURES = fit_logit_transforms(val_cache["logits"], val_cache["labels"])

temperature_df = pd.DataFrame({
    "run_id": [spec["run_id"] for spec in MODEL_SPECS],
    "temperature": TEMPERATURES.reshape(-1),
    "z_mean": Z_MEAN.reshape(-1),
    "z_std": Z_STD.reshape(-1),
})
temperature_df.to_csv(TEMPERATURES_CSV, index=False)
display(temperature_df)


def transformed_logits(cache, transform_name):
    logits = cache["logits"]
    if transform_name == "raw":
        return logits
    if transform_name == "zscore":
        return (logits - Z_MEAN) / Z_STD
    if transform_name == "temperature":
        return logits / TEMPERATURES
    raise ValueError(transform_name)

## 5. Logit Ensemble Search

In [ ]:
def probability_vote(probabilities, weights=None):
    if weights is None:
        combined = probabilities.mean(axis=0)
    else:
        combined = np.average(probabilities, axis=0, weights=weights)
    return combined.argmax(axis=1)


def logit_vote(logits, weights=None):
    if weights is None:
        combined = logits.mean(axis=0)
    else:
        combined = np.average(logits, axis=0, weights=weights)
    return combined.argmax(axis=1)


def optimize_weights_for_logits(logits, labels, base_weights):
    candidates = [np.ones(len(base_weights)) / len(base_weights), base_weights / base_weights.sum()]
    candidates.extend(rng.dirichlet(np.ones(len(base_weights))) for _ in range(MODEL_WEIGHT_RANDOM_TRIALS))

    best_weights, best_score = candidates[0], -1.0
    for candidate in candidates:
        score = metric_dict(labels, logit_vote(logits, weights=candidate))["macro_f1"]
        if score > best_score:
            best_weights, best_score = candidate, score
    return best_weights


def optimize_weights_for_probabilities(probabilities, labels, base_weights):
    candidates = [np.ones(len(base_weights)) / len(base_weights), base_weights / base_weights.sum()]
    candidates.extend(rng.dirichlet(np.ones(len(base_weights))) for _ in range(MODEL_WEIGHT_RANDOM_TRIALS))

    best_weights, best_score = candidates[0], -1.0
    for candidate in candidates:
        score = metric_dict(labels, probability_vote(probabilities, weights=candidate))["macro_f1"]
        if score > best_score:
            best_weights, best_score = candidate, score
    return best_weights


def serialize(values):
    return "" if values is None else json.dumps([round(float(value), 8) for value in values])


def evaluate_validation_combo(indices):
    labels = val_cache["labels"]
    model_names = [MODEL_SPECS[index]["run_id"] for index in indices]
    base_weights = np.array([MODEL_SPECS[index]["base_weight"] for index in indices], dtype=float)
    base_weights /= base_weights.sum()

    probabilities = val_cache["probabilities"][list(indices)]
    raw_logits = transformed_logits(val_cache, "raw")[list(indices)]
    z_logits = transformed_logits(val_cache, "zscore")[list(indices)]
    temp_logits = transformed_logits(val_cache, "temperature")[list(indices)]

    opt_prob_weights = optimize_weights_for_probabilities(probabilities, labels, base_weights)
    opt_raw_weights = optimize_weights_for_logits(raw_logits, labels, base_weights)
    opt_z_weights = optimize_weights_for_logits(z_logits, labels, base_weights)
    opt_temp_weights = optimize_weights_for_logits(temp_logits, labels, base_weights)

    method_predictions = {
        "soft_vote": (probability_vote(probabilities), None, "probability"),
        "weighted_soft_vote": (probability_vote(probabilities, weights=base_weights), base_weights, "probability"),
        "optimized_weighted_soft_vote": (probability_vote(probabilities, weights=opt_prob_weights), opt_prob_weights, "probability"),
        "mean_logits": (logit_vote(raw_logits), None, "raw_logits"),
        "weighted_mean_logits": (logit_vote(raw_logits, weights=base_weights), base_weights, "raw_logits"),
        "optimized_weighted_logits": (logit_vote(raw_logits, weights=opt_raw_weights), opt_raw_weights, "raw_logits"),
        "zscore_mean_logits": (logit_vote(z_logits), None, "zscore_logits"),
        "zscore_weighted_logits": (logit_vote(z_logits, weights=base_weights), base_weights, "zscore_logits"),
        "zscore_optimized_weighted_logits": (logit_vote(z_logits, weights=opt_z_weights), opt_z_weights, "zscore_logits"),
        "temperature_mean_logits": (logit_vote(temp_logits), None, "temperature_logits"),
        "temperature_weighted_logits": (logit_vote(temp_logits, weights=base_weights), base_weights, "temperature_logits"),
        "temperature_optimized_weighted_logits": (logit_vote(temp_logits, weights=opt_temp_weights), opt_temp_weights, "temperature_logits"),
    }

    rows = []
    for method, (prediction, weights, fusion_space) in method_predictions.items():
        result = metric_dict(labels, prediction)
        rows.append({
            "models": ",".join(model_names),
            "num_models": len(model_names),
            "method": method,
            "fusion_space": fusion_space,
            "val_accuracy": result["accuracy"],
            "val_macro_f1": result["macro_f1"],
            "val_kappa": result["kappa"],
            "weights_json": serialize(weights),
        })
    return rows

In [ ]:
validation_rows = []
all_indices = list(range(len(MODEL_SPECS)))

total_combinations = sum(
    1 for size in range(MIN_ENSEMBLE_SIZE, MAX_ENSEMBLE_SIZE + 1)
    for _ in itertools.combinations(all_indices, size)
)
print("Searching combinations:", total_combinations)

for size in range(MIN_ENSEMBLE_SIZE, MAX_ENSEMBLE_SIZE + 1):
    for indices in tqdm(itertools.combinations(all_indices, size), desc=f"Ensembles of {size}"):
        validation_rows.extend(evaluate_validation_combo(indices))

validation_search_df = pd.DataFrame(validation_rows)
validation_search_df.to_csv(VALIDATION_SEARCH_CSV, index=False)

print("Top validation logit/probability ensembles:")
display(validation_search_df.sort_values(
    ["val_macro_f1", "val_kappa", "val_accuracy"],
    ascending=False,
).head(30))

print("Best validation result per method:")
display(
    validation_search_df
    .sort_values(["val_macro_f1", "val_kappa", "val_accuracy"], ascending=False)
    .groupby("method", as_index=False)
    .first()
    .sort_values("val_macro_f1", ascending=False)
)

## 6. Test Evaluation For Validation-Selected Method Finalists

In [ ]:
def deserialize(text):
    return None if pd.isna(text) or text == "" else np.asarray(json.loads(text), dtype=float)


def predict_finalist(row, cache):
    model_names = row["models"].split(",")
    indices = [index for index, spec in enumerate(MODEL_SPECS) if spec["run_id"] in model_names]
    method = row["method"]
    weights = deserialize(row["weights_json"])

    probabilities = cache["probabilities"][indices]
    raw_logits = transformed_logits(cache, "raw")[indices]
    z_logits = transformed_logits(cache, "zscore")[indices]
    temp_logits = transformed_logits(cache, "temperature")[indices]

    if method in ["soft_vote", "weighted_soft_vote", "optimized_weighted_soft_vote"]:
        return probability_vote(probabilities, weights=weights)
    if method in ["mean_logits", "weighted_mean_logits", "optimized_weighted_logits"]:
        return logit_vote(raw_logits, weights=weights)
    if method in ["zscore_mean_logits", "zscore_weighted_logits", "zscore_optimized_weighted_logits"]:
        return logit_vote(z_logits, weights=weights)
    if method in ["temperature_mean_logits", "temperature_weighted_logits", "temperature_optimized_weighted_logits"]:
        return logit_vote(temp_logits, weights=weights)
    raise ValueError(method)


def flatten_class_f1(prefix, labels, predictions):
    report = classification_report(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0,
    )
    return {f"{prefix}_f1_class_{class_id}": report[str(class_id)]["f1-score"] for class_id in range(NUM_CLASSES)}


method_finalists = (
    validation_search_df
    .sort_values(["val_macro_f1", "val_kappa", "val_accuracy"], ascending=False)
    .groupby("method", as_index=False)
    .first()
)

finalist_rows = []
prediction_rows = []
for _, finalist in method_finalists.iterrows():
    test_prediction = predict_finalist(finalist, test_cache)
    test_result = metric_dict(test_cache["labels"], test_prediction)
    row = finalist.to_dict()
    row.update({
        "test_accuracy": test_result["accuracy"],
        "test_macro_f1": test_result["macro_f1"],
        "test_kappa": test_result["kappa"],
    })
    row.update(flatten_class_f1("test", test_cache["labels"], test_prediction))
    finalist_rows.append(row)

    for image_name, true_label, predicted_label in zip(test_cache["names"], test_cache["labels"], test_prediction):
        prediction_rows.append({
            "method": finalist["method"],
            "fusion_space": finalist["fusion_space"],
            "models": finalist["models"],
            "image_name": image_name,
            "true_label": true_label,
            "predicted_label": predicted_label,
        })

finalists_df = pd.DataFrame(finalist_rows).sort_values(
    ["val_macro_f1", "val_kappa", "val_accuracy"],
    ascending=False,
)
finalists_df.to_csv(METHOD_FINALISTS_CSV, index=False)
pd.DataFrame(prediction_rows).to_csv(FINALIST_PREDICTIONS_CSV, index=False)

print("Validation-selected finalist for each method:")
display(finalists_df[[
    "method", "fusion_space", "models", "num_models",
    "val_accuracy", "val_macro_f1", "val_kappa",
    "test_accuracy", "test_macro_f1", "test_kappa",
    "test_f1_class_0", "test_f1_class_1", "test_f1_class_2",
    "test_f1_class_3", "test_f1_class_4",
]])

print("Best validation-selected logit/probability ensemble:")
display(finalists_df.iloc[0].to_frame().T)

try:
    from IPython.display import FileLink
    display(FileLink(VALIDATION_SEARCH_CSV))
    display(FileLink(METHOD_FINALISTS_CSV))
    display(FileLink(INDIVIDUAL_VALIDATION_CSV))
    display(FileLink(FINALIST_PREDICTIONS_CSV))
    display(FileLink(TEMPERATURES_CSV))
except Exception:
    print(VALIDATION_SEARCH_CSV)
    print(METHOD_FINALISTS_CSV)
    print(INDIVIDUAL_VALIDATION_CSV)
    print(FINALIST_PREDICTIONS_CSV)
    print(TEMPERATURES_CSV)

## Interpretation Notes

- `mean_logits` averages raw model outputs before softmax.
- `zscore_*` variants fit one global mean/std per model on validation logits and reuse those statistics on test logits.
- `temperature_*` variants fit one scalar temperature per model on validation negative log-likelihood and reuse it on test logits.
- Optimized weights are selected only on validation Macro F1.
- The final test table should be read as method-finalist evaluation, not as another tuning loop.